# Hybrid CCTV Real-Time Anomaly Detection

Notebook ini adalah versi gabungan terbaik dari dua program sebelumnya:
- memakai model autoencoder terlatih dari folder `cctv-video-anomaly-detection-main`;
- tetap bisa real-time dari webcam, file video, atau RTSP CCTV;
- menyimpan hasil deteksi anomali sebagai foto frame per kejadian;
- memakai human detection, tracking sederhana, zona alert, dan filter durasi seperti prinsip PASS-CCTV;
- semua kejadian seperti vandalism, abusive/violence/perkelahian, dan pencurian dimasukkan ke satu kategori umum: `ANOMALY`.

Output label program hanya:
- `NORMAL`
- `ANOMALY`

## 1. Import Aplikasi Hybrid

In [ ]:
from pathlib import Path

from hybrid_realtime_anomaly_app import (
    ANOMALY_CATEGORY,
    DEFAULT_MODEL_PATH,
    DEFAULT_OUTPUT_DIR,
    RuntimeConfig,
    calibrate_normal,
    run,
)

print("Model path:", DEFAULT_MODEL_PATH)
print("Model tersedia:", DEFAULT_MODEL_PATH.exists())
print("Output folder:", DEFAULT_OUTPUT_DIR)
print("Kategori output:", ANOMALY_CATEGORY["label"])
print("Contoh kejadian yang masuk ANOMALY:")
for event in ANOMALY_CATEGORY["included_events"]:
    print("-", event)

## 2. Kalibrasi Normal Webcam

Jalankan ini sebelum deteksi webcam jika kamera diam masih sering dianggap anomali. Saat kalibrasi, arahkan webcam ke kondisi normal dan jangan lakukan gerakan mencurigakan.

In [ ]:
calibration_config = RuntimeConfig(
    source=0,
    model_path=DEFAULT_MODEL_PATH,
    output_dir=DEFAULT_OUTPUT_DIR,
    show_window=True,
    save_video=False,
)

# Jalankan untuk merekam baseline normal webcam.
# calibration_path = calibrate_normal(calibration_config, frames=300, percentile_value=99.5, safety_scale=1.25)
# print("Gunakan file kalibrasi ini:", calibration_path)

## 3. Jalankan dengan Webcam Laptop

Gunakan cell ini untuk real-time dari webcam laptop. Jika sudah melakukan kalibrasi, aktifkan `calibration_file=DEFAULT_OUTPUT_DIR / "normal_calibration.json"`.

In [ ]:
webcam_config = RuntimeConfig(
    source=0,
    model_path=DEFAULT_MODEL_PATH,
    output_dir=DEFAULT_OUTPUT_DIR,
    calibration_file=DEFAULT_OUTPUT_DIR / "normal_calibration.json",
    save_stride=1,
    show_window=True,
    save_video=True,
)

# Jalankan untuk mulai deteksi webcam.
# run(webcam_config)

## 4. Jalankan dengan Footage CCTV

Gunakan cell ini jika sumbernya file video, misalnya `data/cctv.mp4`.

In [ ]:
video_config = RuntimeConfig(
    source="data/cctv.mp4",
    model_path=DEFAULT_MODEL_PATH,
    output_dir=DEFAULT_OUTPUT_DIR,
    save_stride=1,
    show_window=True,
    save_video=True,
)

if not Path(video_config.source).exists():
    raise FileNotFoundError("File data/cctv.mp4 tidak ditemukan. Pastikan video sudah berada di folder data/.")

# Jalankan untuk mulai deteksi footage CCTV.
# run(video_config)

## 5. Mode Zona Alert dan Tracking Manusia

Mode ini mengikuti ide dari jurnal PASS-CCTV: sistem tidak hanya melihat frame aneh, tetapi juga mendeteksi manusia, melacak ID orang, dan menghitung berapa lama orang berada di zona penting. Contoh zona ada di `zones/alert_zone_example.json`.

In [ ]:
zone_config = RuntimeConfig(
    source="data/cctv.mp4",
    model_path=DEFAULT_MODEL_PATH,
    output_dir=DEFAULT_OUTPUT_DIR,
    zone_file=Path("zones/alert_zone_example.json"),
    zone_loiter_frames=75,
    save_stride=1,
    show_window=True,
    save_video=True,
)

# Jalankan untuk deteksi dengan zona alert dan tracking manusia.
# run(zone_config)

## 6. Jalankan Tanpa Window Video

Mode ini cocok jika `cv2.imshow` bermasalah. Program tetap menyimpan foto frame anomali, log CSV, dan video output.

In [ ]:
headless_config = RuntimeConfig(
    source="data/cctv.mp4",
    model_path=DEFAULT_MODEL_PATH,
    output_dir=DEFAULT_OUTPUT_DIR,
    save_stride=1,
    show_window=False,
    save_video=True,
)

# Jalankan jika tidak ingin membuka window video.
# run(headless_config)

## 7. Lokasi Output

Setelah program dijalankan, hasil akan tersimpan di:
- `hybrid_outputs/anomaly_frames/` untuk foto frame anomali;
- `hybrid_outputs/anomaly_log_v2.csv` untuk log deteksi lengkap;
- `hybrid_outputs/videos/` untuk video hasil overlay;
- `hybrid_outputs/anomaly_definition.json` untuk definisi kategori anomali.